<a href="https://colab.research.google.com/github/rakelcey/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# County Rainfall and Planting Advisory

Import Libraries and Load Dataset

In [1]:
import numpy as np
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print("Dataset loaded successfully!")

Dataset loaded successfully!


Clean Machakos Data

In [2]:
def clean_machakos(raw_data, months):
    valid_vals = []
    parsed = []
    imputed_months = []

    for m, val in zip(months, raw_data):
        try:
            # Strip whitespace and convert to int
            num = int(val.strip())
            parsed.append(num)
            valid_vals.append(num)
        except ValueError:
            parsed.append(None)
            imputed_months.append(m)

    avg_val = round(sum(valid_vals) / len(valid_vals))

    cleaned_machakos = []
    for val in parsed:
        if val is None:
            cleaned_machakos.append(avg_val)
        else:
            cleaned_machakos.append(val)

    print(f"Imputed months for Machakos (using average value {avg_val}): {imputed_months}")
    return cleaned_machakos

machakos_clean = clean_machakos(machakos_raw, month_names)

Imputed months for Machakos (using average value 68): ['Apr', 'Sep']


Combine into a NumPy Array

In [3]:
all_data = clean_rows + [machakos_clean]
rainfall_array = np.array(all_data)

print(f"Rainfall Array Shape: {rainfall_array.shape}")
print(f"Rainfall Array Dtype: {rainfall_array.dtype}")

Rainfall Array Shape: (5, 12)
Rainfall Array Dtype: int64


Annual Totals, Monthly Averages, and Wettest Months

In [4]:
# Annual totals per county
annual_totals = np.sum(rainfall_array, axis=1)

# Average rainfall per month across all counties
monthly_averages = np.mean(rainfall_array, axis=0)

# Wettest month per county
wettest_indices = np.argmax(rainfall_array, axis=1)
wettest_months = [month_names[i] for i in wettest_indices]

print("Annual Totals (mm):", dict(zip(counties, annual_totals)))
print("\nWettest Month per County:")
for c, wm in zip(counties, wettest_months):
    print(f"- {c}: {wm}")

Annual Totals (mm): {'Nairobi': np.int64(1045), 'Nakuru': np.int64(916), 'Kisumu': np.int64(1371), 'Mombasa': np.int64(1075), 'Machakos': np.int64(811)}

Wettest Month per County:
- Nairobi: Apr
- Nakuru: Apr
- Kisumu: Apr
- Mombasa: May
- Machakos: Nov


Dry Months and Recommended Planting Windows

In [5]:
# Dry months count (< 50 mm)
dry_mask = rainfall_array < 50
dry_counts = np.sum(dry_mask, axis=1)

# Planting window (highest consecutive 2-month sum)
consecutive_sums = rainfall_array[:, :-1] + rainfall_array[:, 1:]
best_window_indices = np.argmax(consecutive_sums, axis=1)

planting_windows = []
for idx in best_window_indices:
    m1 = month_names[idx]
    m2 = month_names[idx + 1]
    planting_windows.append(f"{m1}-{m2}")

print("Dry Month Counts:", dict(zip(counties, dry_counts)))
print("\nRecommended Planting Windows:")
for c, pw in zip(counties, planting_windows):
    print(f"- {c}: {pw}")

Dry Month Counts: {'Nairobi': np.int64(4), 'Nakuru': np.int64(3), 'Kisumu': np.int64(0), 'Mombasa': np.int64(2), 'Machakos': np.int64(6)}

Recommended Planting Windows:
- Nairobi: Apr-May
- Nakuru: Apr-May
- Kisumu: Apr-May
- Mombasa: Apr-May
- Machakos: Nov-Dec


Generate CSV Summary

In [6]:
import csv

csv_filename = "rainfall_summary.csv"
with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # Write header
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    # Write rows
    for c, total, wm, dc, pw in zip(counties, annual_totals, wettest_months, dry_counts, planting_windows):
        writer.writerow([c, total, wm, dc, pw])

print(f"File '{csv_filename}' successfully created and written!")

File 'rainfall_summary.csv' successfully created and written!


## Planting Advisory
Farmer-support teams should advise farmers in Nairobi, Nakuru, Kisumu, and Machakos to prepare their fields and plant crops ahead of their primary wet seasons, typically spanning March to May. In Mombasa, planting should align with the April–May or October–November secondary wet windows, ensuring proper soil moisture retention during dry intervals. Advisory SMS broadcasts must emphasize monitoring local precipitation trends closely, as dry-month counts vary significantly across counties.

## Data Quality Notes
The Machakos weather station dataset contained missing values, textual markers ("N/A"), and whitespace anomalies that required programmatic exception handling. These invalid entries were successfully isolated using a try/except ValueError block and imputed using the rounded average of valid monthly readings. Stakeholders should note that imputed values represent mathematical estimations rather than direct physical observations, which introduces minor uncertainty into Machakos's annual totals.